In [46]:
# Install required libraries for Kaggle
# !pip install boto3 sentence-transformers transformers requests -q

import os
import shutil
import glob
import json
import requests
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import numpy as np
from collections import Counter
import re
import warnings

# Suppress verbose HuggingFace warnings for a cleaner output
warnings.filterwarnings("ignore")

# Kaggle-specific working directories
WORK_DIR = '/kaggle/working/'
DATA_DIR = os.path.join(WORK_DIR, 'pmc_articles')

# Teardown phase: Destroy the directory if it exists from a previous run
if os.path.exists(DATA_DIR):
    print("Purging old storage...")
    shutil.rmtree(DATA_DIR)
    
os.makedirs(DATA_DIR, exist_ok=True)

# Configure anonymous access to S3
s3_client = boto3.client(
    's3', 
    region_name='us-east-1', 
    config=Config(signature_version=UNSIGNED)
)
BUCKET_NAME = 'pmc-oa-opendata'

print("Dependencies loaded and environment configured.")

Purging old storage...
Dependencies loaded and environment configured.


In [47]:
def fetch_pmcids_from_api(query, max_results=5):
    """Tier 1: Uses the NCBI API to find relevant PMCIDs without downloading text."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pmc", "term": query, "retmode": "json", "retmax": max_results}
    
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        pmid_list = response.json().get("esearchresult", {}).get("idlist", [])
        return [f"PMC{pmid}" for pmid in pmid_list]
    return []

def download_articles(pmcid_list, max_downloads=5):
    """Tier 2: Downloads articles from S3 until the target cap is reached."""
    downloaded_paths = []
    
    for pmcid in pmcid_list:
        # --- THE FIX: Stop the loop once we hit our cap ---
        if len(downloaded_paths) >= max_downloads:
            break 
            
        s3_key = f"deprecated/oa_comm/txt/all/{pmcid}.txt" 
        local_path = os.path.join(DATA_DIR, f"{pmcid}.txt")
        
        if not os.path.exists(local_path):
            try:
                response = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
                with open(local_path, 'wb') as f:
                    f.write(response['Body'].read())
                downloaded_paths.append(local_path)
            except Exception:
                pass # Skip silently if file is missing in S3
        else:
            downloaded_paths.append(local_path)
            
    return downloaded_paths

def clean_full_text(text):
    """Cleans the text and aggressively filters out academic boilerplate."""
    paragraphs = text.split('\n\n')
    valid_paragraphs = []
    
    # The Boilerplate Blacklist
    blacklist = [
        "creative commons", "copyright", "conflict of interest", 
        "competing interest", "correspondence", "author contribution",
        "none declared", "plos policies", "publisher's note",
        "peer review", "open access", "submitted to", "funding"
    ]
    
    for p in paragraphs:
        clean_p = " ".join(p.split())
        contains_boilerplate = any(bad in clean_p.lower() for bad in blacklist)
        
        if len(clean_p) > 200 and not contains_boilerplate:
            valid_paragraphs.append(clean_p)
            
    return valid_paragraphs

print("Data Ingestion functions are ready.")

Data Ingestion functions are ready.


In [48]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("Loading Retriever/Extractor Model (all-MiniLM)...")
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') 

print("Loading Summarizer Model (facebook/bart-large-cnn)...")
summarizer_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(summarizer_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_name)

print("\nAll AI Models successfully loaded into memory.")

Loading Retriever/Extractor Model (all-MiniLM)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Summarizer Model (facebook/bart-large-cnn)...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]


All AI Models successfully loaded into memory.


In [51]:
from sklearn.metrics.pairwise import cosine_similarity

def extractor_agent(query, document_dicts):
    """Embeds paragraphs dynamically and extracts all chunks above the relevance threshold."""
    query_embedding = embedder.encode([query])
    extracted_docs = []
    
    for doc in document_dicts:
        if not doc["paragraphs"]: continue
            
        para_embeddings = embedder.encode(doc["paragraphs"])
        similarities = cosine_similarity(query_embedding, para_embeddings)[0]
        
        # Find the single best paragraph matching the query within this document
        best_idx = np.argmax(similarities)
        best_score = similarities[best_idx]
        
        # NEW REQUIREMENT: Only keep it if the resemblance is over 40% (0.40)
        if best_score >= 0.40:
            extracted_docs.append({
                "pmcid": doc["pmcid"],
                "relevance_score": best_score,
                "text": doc["paragraphs"][best_idx]
            })
            
    # Sort by relevance, highest to lowest
    extracted_docs = sorted(extracted_docs, key=lambda x: x["relevance_score"], reverse=True)
    
    # NEW REQUIREMENT: Return all matching documents, do not limit to top 2
    return extracted_docs

def extract_keywords(text, num_keywords=5):
    """Term-frequency keyword extraction."""
    words = re.findall(r'\b[a-z]{4,}\b', text.lower())
    stop_words = {'that', 'with', 'from', 'this', 'were', 'which', 'their', 'have', 'been', 'will', 'also', 'than', 'these'}
    filtered_words = [w for w in words if w not in stop_words]
    most_common = Counter(filtered_words).most_common(num_keywords)
    return [word for word, count in most_common]

def summarizer_agent(extracted_docs):
    """Generates concise scientific summaries using BART."""
    summarized_results = []
    for doc in extracted_docs:
        input_text = doc["text"][:1024] 
        inputs = sum_tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)
        
        summary_ids = sum_model.generate(
            inputs.input_ids, max_length=75, min_length=25, 
            length_penalty=2.0, num_beams=4, early_stopping=True
        )
        
        summary_text = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        keywords = extract_keywords(doc["text"])
        
        doc["summary"] = summary_text
        doc["keywords"] = keywords
        summarized_results.append(doc)
        
    return summarized_results

def verifier_agent(summarized_docs):
    """Categorizes summaries using fast embedding distance instead of heavy NLI inference."""
    THEMES = ["Artificial Intelligence and Machine Learning", "Clinical Trials and Efficacy", "Standard Medical Procedures"]
    verified_results = []
    
    # Embed the target themes (using the 'embedder' already loaded in Cell 3)
    theme_embeddings = embedder.encode(THEMES)
    
    for doc in summarized_docs:
        # Embed the generated summary
        summary_embedding = embedder.encode([doc["summary"]])
        
        # Calculate cosine similarity against all themes
        similarities = cosine_similarity(summary_embedding, theme_embeddings)[0]
        
        # Find the index of the highest similarity score
        best_idx = np.argmax(similarities)
        best_score = similarities[best_idx]
        
        # Format the output to match the original pipeline structure
        doc["theme"] = THEMES[best_idx]
        doc["theme_confidence"] = float(best_score) 
        verified_results.append(doc)
        
    return verified_results

print("Multi-Agent logic updated with new thresholds.")

Multi-Agent logic updated with new thresholds.


In [52]:
def run_case_study_pipeline(query, target_documents=3):
    print(f"\n{'='*60}\nQUERY: '{query}'\n{'='*60}")
    
    # 1. API Fetch & Download
    print("1. [Data Tier] Querying NCBI API & Downloading S3 files...")
    target_pmcids = fetch_pmcids_from_api(query, max_results=30) 
    file_paths = download_articles(target_pmcids, max_downloads=target_documents)
    
    print(f"   -> Successfully downloaded {len(file_paths)} files from S3.")
    
    # Parse downloaded files
    documents = []
    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            pmcid = os.path.basename(path).replace('.txt', '')
            paragraphs = clean_full_text(f.read())
            documents.append({"pmcid": pmcid, "paragraphs": paragraphs})
    
    # 2. Extractor
    print("2. [Extractor Agent] Scanning documents for ->40% semantic match...")
    extracted_docs = extractor_agent(query, documents)
    
    if not extracted_docs:
        print("No passages met the 40% relevance threshold.")
        return
        
    extracted_docs = extracted_docs[:target_documents]
    print(f"   -> Found {len(extracted_docs)} highly relevant passages.")
    
    # 3. Summarizer
    print("3. [Summarizer Agent] Compressing passages and generating keywords...")
    summarized_docs = summarizer_agent(extracted_docs)
    
    # 4. Verifier
    print("4. [Verifier Agent] Categorizing outputs against case-study themes...")
    final_results = verifier_agent(summarized_docs)
    
    # Output
    for i, res in enumerate(final_results, 1):
        print(f"\n--- Result {i} (PMCID: {res['pmcid']}) ---")
        print(f"Match Score: {res['relevance_score']:.2f}")
        print(f"Theme: {res['theme']}")
        print(f"Keywords: {', '.join(res['keywords'])}")
        print(f"Summary: {res['summary']}")

In [53]:
# The specific test queries mandated by the case study
test_queries = [
    "Adverse events with mRNA vaccines in pediatrics",
    "Transformer-based models for protein folding",
    "Clinical trial outcomes for monoclonal antibodies in oncology"
]

for q in test_queries:
    # Set target_documents to strictly cap the processing load per query
    run_case_study_pipeline(q, target_documents=4)


QUERY: 'Adverse events with mRNA vaccines in pediatrics'
1. [Data Tier] Querying NCBI API & Downloading S3 files...
   -> Successfully downloaded 0 files from S3.
2. [Extractor Agent] Scanning documents for ->40% semantic match...
No passages met the 40% relevance threshold.

QUERY: 'Transformer-based models for protein folding'
1. [Data Tier] Querying NCBI API & Downloading S3 files...
   -> Successfully downloaded 4 files from S3.
2. [Extractor Agent] Scanning documents for ->40% semantic match...
   -> Found 3 highly relevant passages.
3. [Summarizer Agent] Compressing passages and generating keywords...
4. [Verifier Agent] Categorizing outputs against case-study themes...

--- Result 1 (PMCID: PMC13262399) ---
Match Score: 0.50
Theme: Artificial Intelligence and Machine Learning
Keywords: flexibility, residue, level, analysis, rmsf
Summary: Residue-level flexibility analysis (RMSF) showed localized increases in flexibility in regions surrounding the ligand-binding pocket. Most pro

In [54]:
# Extended test queries to evaluate different pipeline capabilities
more_test_queries = [
        "Graph neural networks for targeted drug discovery and repurposing", 
        "CRISPR-Cas9 off-target mutations in human gene therapy", 
        "Long-term neurological impacts of COVID-19 in young adults", 
        "Gut microbiome influence on major depressive disorder", 
        "Microplastic accumulation in human cardiovascular tissue" 
]

for q in more_test_queries:
    # Setting target_documents to 3 or 4 keeps the test run relatively quick
    run_case_study_pipeline(q, target_documents=3)


QUERY: 'Graph neural networks for targeted drug discovery and repurposing'
1. [Data Tier] Querying NCBI API & Downloading S3 files...
   -> Successfully downloaded 3 files from S3.
2. [Extractor Agent] Scanning documents for ->40% semantic match...
   -> Found 2 highly relevant passages.
3. [Summarizer Agent] Compressing passages and generating keywords...
4. [Verifier Agent] Categorizing outputs against case-study themes...

--- Result 1 (PMCID: PMC13264839) ---
Match Score: 0.54
Theme: Artificial Intelligence and Machine Learning
Keywords: models, graph, such, patient, relationships
Summary: Graph-based deep learning models use Graph Convolutional Networks (GCNs) to improve classification accuracy for disease subtyping, diagnosis, and identification of new targets. These models typically present multiomics data as interconnected graphs, where nodes represent genes or samples and edges capture the biological relationships or similarity.

--- Result 2 (PMCID: PMC13260815) ---
Match Sc